# Этап 4. Прототип RAG по ПУЭ: точность поиска (редакция v6.3, проект V2)

**Тема:** «Разработка интеллектуального Telegram-бота на основе языковой модели для консультаций по ПУЭ». Репозиторий: https://github.com/kirag-ozyaz/Diplom_AI

Ноутбук строит все таблицы и интерактивные графики отчёта `Otchet_etap4_v6.3.docx` прямо из файлов результатов. Он читает:

- **V2:** все `metrics.json` в `V2/results/retrieval/split*/…` и набор вопросов `V2/data/rag_eval_questions/eval_questions_split.json`;
- **генерация и оценка (v6.3):** `V2/results/generation/answers/*_rerank50/`, `judge/*_rerank50/` (судья) и `answers/*_refusal/`, `judge/*_refusal/` (вопросы без ответа в ПУЭ), раздел 6;
- **первый проект (v5):** `scripts/stage4_eval/results/retrieval/stage4_rank_results_{base,e5,rerank}.json`.

Milvus, модели и видеокарта не нужны. Опыт определяется по настройкам внутри `metrics.json`, поэтому имена папок прогонов не важны; если прогонов одного опыта несколько, берётся последний. Повторный прогон поиска (раздел 7) выключен флагом `RUN_LIVE = False`, пересчёт метрик из сохранённой выдачи — флагом `RUN_EVAL = False`.

## 0. Подготовка

In [ ]:
RUN_LIVE = False   # True — заново выполнить поиск (нужен Milvus и GPU), раздел 7
RUN_EVAL = False   # True — пересчитать metrics.json из сохранённых retrieval.json без Milvus, раздел 7

import json
from pathlib import Path
from collections import Counter
import plotly.graph_objects as go
from IPython.display import Markdown, display

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "V2" / "results" / "retrieval").is_dir():
            return p
    raise FileNotFoundError("Не найден корень проекта (нет V2/results/retrieval). Откройте ноутбук внутри проекта.")

ROOT = find_root()
V2RES = ROOT / "V2" / "results" / "retrieval"
V1RES = ROOT / "scripts" / "stage4_eval" / "results" / "retrieval"
QFILE = ROOT / "V2" / "data" / "rag_eval_questions" / "eval_questions_split.json"
print("Корень проекта:", ROOT)

def md_table(head, rows):
    s = "| " + " | ".join(map(str, head)) + " |\n|" + "---|" * len(head) + "\n"
    s += "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)
    display(Markdown(s))

def pct(x): return f"{x * 100:.1f}"

EXP = ["o1", "o2", "o3", "o4", "o5"]
NAME = {"o1": "Опыт 1. Без префиксов", "o2": "Опыт 2. Префиксы E5", "o3": "Опыт 3. Реранкер, 20",
        "o4": "Опыт 4. Реранкер, 50", "o5": "Опыт 5. Реранкер, 50 + CLIP", "ctl": "Контроль (повтор опыта 2)"}
COLOR = {"o1": "#9aa5b1", "o2": "#5b8def", "o3": "#f2a541", "o4": "#e4572e", "o5": "#2a9d8f", "ctl": "#7b61ff"}

def bar(title, cats, series, ytitle="%", yrange=None, fmt=".1f"):
    fig = go.Figure()
    for name, vals, color in series:
        fig.add_bar(name=name, x=cats, y=vals, marker_color=color, text=[format(v, fmt) for v in vals],
                    textposition="outside", hovertemplate="%{x}<br>" + name + ": %{y:" + fmt + "}<extra></extra>")
    fig.update_layout(title=title, barmode="group", yaxis_title=ytitle, height=430,
                      legend=dict(orientation="h", y=-0.2), margin=dict(t=60, b=40))
    if yrange: fig.update_yaxes(range=yrange)
    fig.show()

## 1. Загрузка результатов V2

In [ ]:
def classify(m):
    s = (m.get("search_settings") or {}).get("mode") or {}
    if m.get("clip_enabled"): return "o5"
    if s.get("rerank_model"): return "o4" if s.get("candidate_depth", 0) >= 50 else "o3"
    if s.get("query_prefix", "") == "": return "o1"
    return "ctl" if s.get("collection", "").endswith("_e5") else "o2"

RUNS = {}
for f in sorted(V2RES.glob("split*/**/metrics.json")):
    m = json.loads(f.read_text(encoding="utf-8"))
    if m.get("qrels_version") != "split-720-exact-text-v1" or len(m.get("questions", [])) != 720:
        continue
    e = classify(m)
    if e not in RUNS or m["created_at"] > RUNS[e][1]["created_at"]:
        RUNS[e] = (f, m)
missing = [NAME[e] for e in EXP if e not in RUNS]
assert not missing, f"Нет прогонов: {missing}"
md_table(["Опыт", "Коллекция", "Кандидатов", "CLIP", "Создан", "Файл"],
         [[NAME[e], RUNS[e][1]["search_settings"]["mode"]["collection"],
           (50 if RUNS[e][1].get("clip_enabled") else RUNS[e][1]["search_settings"]["mode"].get("candidate_depth", "—")),
           "да" if RUNS[e][1].get("clip_enabled") else "нет", RUNS[e][1]["created_at"][:16].replace("T", " "),
           f"`{RUNS[e][0].relative_to(ROOT).as_posix()}`"] for e in [*EXP, "ctl"] if e in RUNS])

D = {e: {r["qid"]: r for r in RUNS[e][1]["questions"]} for e in RUNS}
QS = {q["id"]: q for q in json.loads(QFILE.read_text(encoding="utf-8"))["questions"]}
GROUPS = {"orig": lambda q: "." not in q, "parts": lambda q: "." in q, "all": lambda q: True}
GN = {"orig": "268 исходных", "parts": "452 части", "all": "все 720"}

def agg(e, group, k, level="content", key="hit"):
    rows = [r for q, r in D[e].items() if GROUPS[group](q)]
    if level == "facts":
        return sum(1 for r in rows if r["metrics"][k]["facts"][key]) / len(rows)
    return sum(r["metrics"][k][level][key] or 0 for r in rows) / len(rows)

## 2. База данных и набор вопросов

In [ ]:
qs = list(QS.values())
orig = [q for q in qs if q["is_original_question"]]
sec = Counter(q["expected_clause"].split(".")[0] for q in orig)
chunks = ROOT / "V2" / "data" / "chunked_merged_manual"
rows = [["Вопросов всего", len(qs)], ["Исходных (дословно из eval_questions.json)", len(orig)], ["Частей X.1, X.2 …", len(qs) - len(orig)]]
if chunks.is_dir():
    recs = [json.loads(l) for f in chunks.glob("*.chunked.jsonl") for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]
    rows += [["Файлов чанков", len(list(chunks.glob("*.chunked.jsonl")))], ["Чанков", len(recs)],
             ["С номером пункта", sum(1 for r in recs if r["metadata"].get("Clause"))],
             ["С таблицами", sum(1 for r in recs if r["metadata"].get("contains_tables"))],
             ["Со ссылками на изображения", sum(1 for r in recs if r["metadata"].get("contains_images"))]]
md_table(["Показатель", "Значение"], rows)
md_table(["Раздел ПУЭ", "Исходных вопросов"], [[s, sec[s]] for s in sorted(sec)])

## 3. Параметры опытов (из metrics.json)

In [ ]:
def p(e):
    m = RUNS[e][1]; s = m["search_settings"]["mode"]; c = m["search_settings"].get("clip") if m.get("clip_enabled") else None
    return [NAME[e], s["collection"], s["model"], f'«{s.get("query_prefix", "")}» / «{s.get("passage_prefix", "")}»',
            s.get("rerank_model", "нет"), 50 if c else s.get("candidate_depth", "—"),
            f'{c["model"]}, RRF k={c["rrf_k"]}' if c else "нет", m["resources_mode"]]
md_table(["Опыт", "Коллекция", "Модель", "Префиксы", "Реранкер", "Кандидатов", "CLIP", "Зачёт ресурсов"], [p(e) for e in [*EXP, "ctl"] if e in RUNS])

## 5. Метрики

### 5.1–5.2. Итоговые таблицы

Hit@k — доля вопросов, у которых среди первых k есть чанк с дословным эталонным текстом. MRR — по выдаче глубиной 20. «Все факты в топ-5» — все обязательные абзацы эталона в первых пяти чанках.

In [ ]:
for g in ("orig", "all", "parts"):
    display(Markdown(f"**{GN[g]}**"))
    md_table(["Опыт", "Hit@1", "Hit@3", "Hit@5", "Hit@10", "Hit@20", "MRR", "nDCG@5", "Все факты в топ-5"],
             [[NAME[e]] + [pct(agg(e, g, k)) for k in ("1", "3", "5", "10", "20")]
              + [f'{agg(e, g, "20", key="rr"):.3f}', f'{agg(e, g, "5", key="ndcg_reference_set"):.3f}',
                 pct(agg(e, g, "5", "facts", "full_text_answer"))] for e in [*EXP, "ctl"] if e in RUNS])

In [ ]:
bar("Рисунок 1 — Hit@k, 268 исходных вопросов", ["Hit@1", "Hit@3", "Hit@5", "Hit@10"],
    [(NAME[e], [agg(e, "orig", k) * 100 for k in ("1", "3", "5", "10")], COLOR[e]) for e in EXP], yrange=[50, 105])

### 5.3. MRR и nDCG@5

In [ ]:
bar("Рисунок 2 — MRR и nDCG@5, 268 исходных вопросов", [NAME[e] for e in EXP],
    [("MRR (глубина 20)", [agg(e, "orig", "20", key="rr") for e in EXP], "#5b8def"),
     ("nDCG@5", [agg(e, "orig", "5", key="ndcg_reference_set") for e in EXP], "#2a9d8f")], ytitle="", yrange=[0.7, 1.0], fmt=".3f")

### 5.4. Исходные вопросы и их части

In [ ]:
bar("Рисунок 3 — Hit@1: исходные вопросы и части", [NAME[e] for e in EXP],
    [("исходные (268)", [agg(e, "orig", "1") * 100 for e in EXP], "#5b8def"),
     ("части X.1, X.2 … (452)", [agg(e, "parts", "1") * 100 for e in EXP], "#f2a541")], yrange=[55, 100])
bar("Рисунок 4 — Все обязательные факты в первых пяти фрагментах", [NAME[e] for e in EXP],
    [("исходные (268)", [agg(e, "orig", "5", "facts", "full_text_answer") * 100 for e in EXP], "#5b8def"),
     ("все (720)", [agg(e, "all", "5", "facts", "full_text_answer") * 100 for e in EXP], "#2a9d8f")], yrange=[75, 100])

### 5.5. По разделам ПУЭ

In [ ]:
secs = sorted(sec)
def sec_hit(e, s, k="5"):
    ids = [q for q in D[e] if "." not in q and QS[q]["expected_clause"].split(".")[0] == s]
    return sum(D[e][q]["metrics"][k]["content"]["hit"] for q in ids), len(ids)
md_table(["Раздел", "Вопросов"] + [NAME[e] for e in EXP], [[s, sec[s]] + [sec_hit(e, s)[0] for e in EXP] for s in secs])
bar("Рисунок 5 — Hit@5 по разделам ПУЭ", [f"Раздел {s}" for s in secs],
    [(NAME[e], [sec_hit(e, s)[0] / sec_hit(e, s)[1] * 100 for s in secs], COLOR[e]) for e in EXP], yrange=[60, 102], fmt=".0f")

### 5.6. Первый проект (v5) и V2: было / стало

В первом проекте попаданием считался чанк эталонного пункта, поэтому для V2 здесь взят тот же уровень зачёта — по пункту (`clause`). Вопросы те же 268.

In [ ]:
V1 = {}
for n, suf in ((1, "base"), (2, "e5"), (3, "rerank")):
    f = V1RES / f"stage4_rank_results_{suf}.json"
    if f.is_file(): V1[n] = json.loads(f.read_text(encoding="utf-8"))
if len(V1) < 3:
    print("Файлы первого проекта не найдены в", V1RES, "— раздел пропущен")
else:
    PAIR = [(1, "o1", "без префиксов"), (2, "o2", "префиксы E5"), (3, "o3", "реранкер, 20")]
    md_table(["Опыт", "Hit@1 v5", "Hit@1 V2", "Hit@5 v5", "Hit@5 V2", "Hit@10 v5", "Hit@10 V2", "MRR@10 v5", "MRR@10 V2", "Прогон v5"],
             [[lab, pct(V1[n]["metrics"]["hit@1"]), pct(agg(e, "orig", "1", "clause")), pct(V1[n]["metrics"]["hit@5"]), pct(agg(e, "orig", "5", "clause")),
               pct(V1[n]["metrics"]["hit@10"]), pct(agg(e, "orig", "10", "clause")), f'{V1[n]["metrics"]["mrr"]:.3f}',
               f'{agg(e, "orig", "10", "clause", "rr"):.3f}', V1[n]["evaluated_at"]] for n, e, lab in PAIR])
    for k, kk in (("1", "hit@1"), ("5", "hit@5"), ("10", "hit@10")):
        bar(f"Рисунок 6 — Hit@{k}: первый проект и V2 (зачёт по пункту)", [lab for _, _, lab in PAIR],
            [("первый проект (v5), 2912 чанков", [V1[n]["metrics"][kk] * 100 for n, _, _ in PAIR], "#9aa5b1"),
             ("V2 (v6), 3146 чанков", [agg(e, "orig", k, "clause") * 100 for _, e, _ in PAIR], "#e4572e")], yrange=[50, 102])
    rows = []
    for n, e, lab in PAIR:
        v1 = {str(x["id"]): x["rank"] is not None and x["rank"] <= 5 for x in V1[n]["details"]}
        v2 = {q: bool(D[e][q]["metrics"]["5"]["clause"]["hit"]) for q in v1}
        rows.append([lab, sum(1 for q in v1 if v2[q] and not v1[q]), sum(1 for q in v1 if v1[q] and not v2[q])])
    md_table(["Опыт", "Hit@5: V2 нашёл, v5 нет", "Hit@5: v5 нашёл, V2 нет"], rows)

### 5.7. Попарные изменения между опытами

In [ ]:
def flips(a, b, k):
    ids = [q for q in D[a] if "." not in q]
    fx = sum(1 for q in ids if not D[a][q]["metrics"][k]["content"]["hit"] and D[b][q]["metrics"][k]["content"]["hit"])
    br = sum(1 for q in ids if D[a][q]["metrics"][k]["content"]["hit"] and not D[b][q]["metrics"][k]["content"]["hit"])
    return f"{fx} / {br}"
steps = [("o1", "o2"), ("o2", "o3"), ("o3", "o4"), ("o4", "o5")] + ([("o2", "ctl")] if "ctl" in D else [])
md_table(["Переход", "Hit@1: исправлено / испорчено", "Hit@5: исправлено / испорчено"],
         [[f"{NAME[a]} → {NAME[b]}", flips(a, b, "1"), flips(a, b, "5")] for a, b in steps])

### 5.8. Типовые отказы поиска

In [ ]:
miss = [q for q in D["o5"] if "." not in q and (not D["o5"][q]["metrics"]["5"]["content"]["hit"] or not D["o4"][q]["metrics"]["5"]["content"]["hit"])]
md_table(["№", "Пункт", "Таблицы / рисунки", "Вопрос"] + [NAME[e] for e in EXP],
         [[q, QS[q]["expected_clause"], ", ".join(QS[q]["required_tables"] + QS[q]["required_figures"]) or "—", QS[q]["query"]]
          + ["да" if D[e][q]["metrics"]["5"]["content"]["hit"] else "нет" for e in EXP] for q in sorted(miss, key=int)])

## 6. Генерация ответов по выдаче опыта 4 и их оценка (редакция v6.3)

Ячейки ниже читают результаты скриптов `V2/scripts/generation/`: последний прогон генерации `answers/*_rerank50/`, оценки судьи `judge/*_rerank50/` (раздел 6.4) и набор вопросов без ответа в ПУЭ — `answers/*_refusal/` и `judge/*_refusal/` (раздел 6.5). Модель и Ollama не нужны.

In [ ]:
import re, statistics as st
GEN = ROOT / "V2" / "results" / "generation"
gdirs = sorted((GEN / "answers").glob("*_rerank50"), key=lambda p: p.name) if (GEN / "answers").is_dir() else []
if not gdirs:
    print("Прогон генерации не найден: нет папки V2/results/generation/answers/*_rerank50")
    ANS = []
else:
    GDIR = gdirs[-1]
    GRUN = json.loads((GDIR / "run.json").read_text(encoding="utf-8"))["settings"]
    rows = [json.loads(l) for l in (GDIR / "answers.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    last = {}
    for r in rows: last[str(r["qid"])] = r          # при --resume берётся последняя строка по вопросу
    ANS = [r for r in last.values() if not r.get("error")]
    print("Папка:", GDIR.relative_to(ROOT).as_posix(), "| строк:", len(rows), "| вопросов без ошибки:", len(ANS),
          "| с ошибкой:", sum(1 for r in last.values() if r.get("error")))
    md_table(["Параметр", "Значение"], [
        ["Выдача (вход)", f"`{GRUN['input']}`"], ["Режим поиска", GRUN["retrieval_mode"]],
        ["Кандидатов для реранкера", GRUN["search_settings"]["mode"].get("candidate_depth")],
        ["Модель", GRUN["model"]], ["temperature", GRUN["temperature"]], ["num_predict", GRUN["num_predict"]],
        ["num_ctx", GRUN["num_ctx"]], ["Фрагментов в контексте", GRUN["top_k"]], ["Обрезка фрагмента, символов", GRUN["max_chars"]],
        ["Начало прогона", GRUN["started_at"]]])

In [ ]:
CL = re.compile(r"\b\d\.\d{1,2}\.\d{1,3}\b")
REF = re.compile(r"(в (приведённом|приведенном|предоставленном|данном) контексте (нет|не)|контекст(е)? не содержит|нет (информации|сведений|данных)|информаци\w* (нет|отсутству)|не указан\w* в (контексте|приведённых))", re.I)
def gstat(G):
    n = len(G) or 1
    return {"n": len(G), "el_mean": st.mean(r["elapsed_s"] for r in G), "el_med": st.median(r["elapsed_s"] for r in G),
            "pt": st.mean(r["prompt_tokens"] for r in G), "at": st.mean(r["answer_tokens"] for r in G),
            "clause": sum(bool(CL.search(r["answer"])) for r in G) / n,
            "notctx": sum(bool(set(CL.findall(r["answer"])) - {c["clause"] for c in r["contexts"]}) for r in G),
            "refused": [r["qid"] for r in G if REF.search(r["answer"])],
            "limit": [r["qid"] for r in G if r.get("answer_hit_limit")],
            "trunc": sum(any(c["truncated"] for c in r["contexts"]) for r in G)}
if ANS:
    GROUPS = {"Все": ANS, "Исходные": [r for r in ANS if r["is_original"]], "Части": [r for r in ANS if not r["is_original"]]}
    GS = {k: gstat(v) for k, v in GROUPS.items()}
    md_table(["Показатель", *GS], [
        ["Вопросов", *[s["n"] for s in GS.values()]],
        ["Время ответа, с: среднее", *[f"{s['el_mean']:.1f}" for s in GS.values()]],
        ["Время ответа, с: медиана", *[f"{s['el_med']:.1f}" for s in GS.values()]],
        ["Токенов в запросе (среднее)", *[round(s["pt"]) for s in GS.values()]],
        ["Токенов в ответе (среднее)", *[round(s["at"]) for s in GS.values()]],
        ["Ответ содержит номер пункта, %", *[pct(s["clause"]) for s in GS.values()]],
        ["Пункт не из заголовков поданных фрагментов", *[s["notctx"] for s in GS.values()]],
        ["Явный отказ", *[len(s["refused"]) for s in GS.values()]],
        ["Ответ упёрся в лимит num_predict", *[len(s["limit"]) for s in GS.values()]],
        ["Есть обрезанный фрагмент", *[s["trunc"] for s in GS.values()]]])
    print("Общее время генерации, мин:", round(sum(r["elapsed_s"] for r in ANS) / 60, 1),
          "| скорость, токенов/с:", round(st.mean(r["tokens_per_s"] for r in ANS if r.get("tokens_per_s")), 1))
    print("Явный отказ:", ", ".join(GS["Все"]["refused"]) or "нет")
    print("Упёрлись в лимит:", ", ".join(GS["Все"]["limit"]) or "нет")

In [ ]:
if ANS:
    fig = go.Figure()
    for k, color in [("Исходные", "#e4572e"), ("Части", "#5b8def")]:
        fig.add_histogram(x=[r["elapsed_s"] for r in GROUPS[k]], name=k, marker_color=color, opacity=0.75, xbins=dict(size=1))
    fig.update_layout(title="Время ответа модели на вопрос", barmode="overlay", xaxis_title="секунд", yaxis_title="вопросов",
                      height=420, legend=dict(orientation="h", y=-0.2))
    fig.show()
    fig = go.Figure()
    for k, color in [("Исходные", "#e4572e"), ("Части", "#5b8def")]:
        fig.add_scatter(x=[r["prompt_tokens"] for r in GROUPS[k]], y=[r["elapsed_s"] for r in GROUPS[k]], mode="markers",
                        name=k, marker=dict(color=color, size=5, opacity=0.6), text=[r["qid"] for r in GROUPS[k]],
                        hovertemplate="вопрос %{text}<br>токенов в запросе %{x}<br>%{y:.1f} с<extra></extra>")
    fig.update_layout(title="Длина запроса и время ответа", xaxis_title="токенов в запросе", yaxis_title="секунд", height=420,
                      legend=dict(orientation="h", y=-0.2))
    fig.show()

In [ ]:
if ANS:
    sel = [r for r in ANS if REF.search(r["answer"]) or r.get("answer_hit_limit")]
    md_table(["№", "Вопрос", "Отметка", "Пункты поданных фрагментов", "Начало ответа"],
             [[r["qid"], r["query"], "отказ" if REF.search(r["answer"]) else "лимит", ", ".join(c["clause"] or "—" for c in r["contexts"]),
               r["answer"][:160].replace("\n", " ").replace("|", "/") + "…"] for r in sel])

### 6.4. Оценка ответов LLM-судьёй

Судья (`judge.py`, та же qwen2.5:14b, temperature 0) видит вопрос, эталонные обязательные факты, пять поданных фрагментов и ответ и ставит 0/1/2 по пяти критериям. В таблицах — средний балл / 2 (1 = максимум). Берётся последняя папка `V2/results/generation/judge/*_rerank50/`.

In [ ]:
CRIT = [("facts", "Факты"), ("numbers", "Числа и единицы"), ("exceptions", "Исключения"), ("sources", "Источник"), ("refusal", "Отказ")]
def last_dir(kind, suffix):
    base = GEN / kind
    ds = sorted(base.glob("*_" + suffix), key=lambda p: p.name) if base.is_dir() else []
    return ds[-1] if ds else None
def read_jsonl(p):
    last = {}
    for l in p.read_text(encoding="utf-8").splitlines():
        if l.strip():
            r = json.loads(l); last[str(r["qid"])] = r
    return list(last.values())
JDIR = last_dir("judge", "rerank50")
if JDIR is None or not (JDIR / "judge_metrics.json").is_file():
    print("Оценок судьи нет: не найдена папка V2/results/generation/judge/*_rerank50 с judge_metrics.json"); JM = None; JR = []
else:
    JM = json.loads((JDIR / "judge_metrics.json").read_text(encoding="utf-8"))
    JR = read_jsonl(JDIR / "judgments.jsonl") if (JDIR / "judgments.jsonl").is_file() else []
    print("Папка:", JDIR.relative_to(ROOT).as_posix(), "| судья:", JM["judge_model"], "| оценено:", JM["n_judged"], "из", JM["n_answers"],
          "| сбоев:", len(JM["failed_qids"]), "| завершено:", JM["created_at"])
    G = {k: v for k, v in JM["answerable"].items() if v}
    NAMES = {"all": "Все", "originals": "Исходные", "parts": "Части", "development": "Разработка", "reserved_evaluation": "Отложенные"}
    md_table(["Показатель", *[f"{NAMES.get(k, k)} ({v['n']})" for k, v in G.items()]],
             [[c[1], *[pct(v[c[0]]) for v in G.values()]] for c in CRIT] + [
              ["Итоговый балл", *[pct(v["total"]) for v in G.values()]],
              ["Все критерии = 2", *[pct(v["all_criteria_2"]) for v in G.values()]],
              ["Все критерии ≥ 1", *[pct(v["all_criteria_ge1"]) for v in G.values()]],
              ["Критическая ошибка", *[pct(v["critical_error_rate"]) for v in G.values()]],
              ["Отказ (по судье)", *[pct(v["refused_rate_judge"]) for v in G.values()]],
              ["Эталонный пункт в ответе", *[pct(v["clause_in_answer"]) for v in G.values()]],
              ["Эталонный пункт в фрагментах", *[pct(v["clause_in_context"]) for v in G.values()]]])

In [ ]:
if JM:
    bar("Оценка судьи по критериям, % от максимума", [c[1] for c in CRIT] + ["Итог"],
        [(NAMES[k], [100 * JM["answerable"][k][c] for c in [x[0] for x in CRIT] + ["total"]], None) for k in ("originals", "parts")],
        ytitle="%", yrange=[70, 101])
if JR:
    tot = lambda r: sum(r["judge"][c] for c, _ in CRIT)
    fig = go.Figure()
    for k, flag, color in [("Исходные", True, "#e4572e"), ("Части", False, "#5b8def")]:
        cnt = Counter(tot(r) for r in JR if r["kind"] == "answerable" and r["is_original"] == flag)
        fig.add_bar(x=list(range(11)), y=[cnt.get(s, 0) for s in range(11)], name=k, marker_color=color, text=[cnt.get(s, 0) for s in range(11)])
    fig.update_layout(title="Сумма баллов судьи на ответ (0–10)", xaxis_title="сумма баллов", yaxis_title="ответов",
                      barmode="group", height=420, legend=dict(orientation="h", y=-0.2))
    fig.show()

In [ ]:
if JR:
    crit_rows = [r for r in JR if r["kind"] == "answerable" and r["judge"].get("critical_error")]
    crit_rows.sort(key=lambda r: [int(x) for x in r["qid"].split(".")])
    print("Критические ошибки:", len(crit_rows), "| эталонный пункт не подан поиском:",
          sum(not r["auto"]["required_clause_in_context"] for r in crit_rows))
    md_table(["№", "Группа", "Пункт во фрагментах", "Баллы (ф/ч/и/и/о)", "Отказ", "Комментарий судьи"],
             [[r["qid"], "исходный" if r["is_original"] else "часть", "да" if r["auto"]["required_clause_in_context"] else "нет",
               "/".join(str(r["judge"][c]) for c, _ in CRIT), "да" if r["judge"].get("refused") else "",
               r["judge"].get("comment", "").replace("|", "/").replace("\n", " ")[:220]] for r in crit_rows])

### 6.5. Вопросы без ответа в ПУЭ: правильный отказ

Набор `V2/data/rag_eval_questions/eval_questions_refusal.json` (48 вопросов R01–R48, `answerability = not_in_pue`): другие нормы, новые технологии, смежные темы, вне темы. Поиск — `refusal_search.py` (конфигурация опыта 4), генерация и судья — те же скрипты. Судья ставит `refusal_correct`: 2 — ясный отказ без выдуманной нормы, 1 — отказ с посторонними пунктами, 0 — ответ по существу. Колонка «ручная проверка» — строгая оценка автора отчёта (засчитан только чистый отказ), она задана списком `MANUAL_NOT_OK` ниже.

In [ ]:
MANUAL_NOT_OK = {"R17", "R22", "R26", "R28", "R32"}   # ручная проверка автора 13.11.2026 (см. отчёт, раздел 6.5)
CATN = {"other_norm": "другие нормы", "new_technology": "новые технологии", "partial_context": "смежные темы", "out_of_domain": "вне темы"}
RJDIR, RADIR = last_dir("judge", "refusal"), last_dir("answers", "refusal")
if RJDIR is None or not (RJDIR / "judgments.jsonl").is_file():
    print("Оценок набора на отказ нет: не найдена папка V2/results/generation/judge/*_refusal"); RJ = []
else:
    RM = json.loads((RJDIR / "judge_metrics.json").read_text(encoding="utf-8"))["not_in_pue"]
    RJ = [r for r in read_jsonl(RJDIR / "judgments.jsonl") if r["kind"] == "not_in_pue"]
    RA = {str(r["qid"]): r for r in read_jsonl(RADIR / "answers.jsonl")} if RADIR else {}
    print("Папка:", RJDIR.relative_to(ROOT).as_posix(), "| оценено:", RM["judged"], "из", RM["n"])
    md_table(["Показатель", "Значение"], [
        ["Правильный отказ (судья: 2)", pct(RM["correct_refusal_rate"])], ["Средний refusal_correct / 2", pct(RM["refusal_correct_mean"])],
        ["Ответил вместо отказа (судья: 0)", pct(RM["answered_instead_rate"])], ["Выдумка по судье", pct(RM["hallucination_rate"])],
        ["Фраза отказа (регулярное выражение)", pct(RM["refused_rate_regex"])],
        ["Правильный отказ (ручная проверка)", pct(sum(r["qid"] not in MANUAL_NOT_OK for r in RJ) / len(RJ))]])
    rows = []
    for c, name in CATN.items():
        g = [r for r in RJ if r["category"] == c]
        if g:
            rows.append((f"{name} ({len(g)})", 100 * sum(r["judge"]["refusal_correct"] == 2 for r in g) / len(g),
                         100 * sum(r["qid"] not in MANUAL_NOT_OK for r in g) / len(g)))
    md_table(["Категория", "Судья, %", "Ручная проверка, %"], [[a, f"{b:.1f}", f"{c:.1f}"] for a, b, c in rows])
    bar("Правильный отказ на вопросах без ответа в ПУЭ, %", [r[0] for r in rows],
        [("судья", [r[1] for r in rows], None), ("ручная проверка", [r[2] for r in rows], None)], ytitle="%", yrange=[0, 110], fmt=".0f")

In [ ]:
if RJ:
    sel = [r for r in RJ if r["judge"]["refusal_correct"] < 2 or r["judge"].get("hallucination") or r["qid"] in MANUAL_NOT_OK]
    md_table(["№", "Категория", "Судья", "Выдумка (судья)", "Ручная", "Вопрос", "Начало ответа"],
             [[r["qid"], CATN.get(r["category"], r["category"]), r["judge"]["refusal_correct"], "да" if r["judge"].get("hallucination") else "",
               "нет" if r["qid"] in MANUAL_NOT_OK else "да", RA.get(r["qid"], {}).get("query", ""),
               RA.get(r["qid"], {}).get("answer", "")[:200].replace("\n", " ").replace("|", "/") + "…"] for r in sel])

## 7. Команды прогонов и повторный расчёт

Команды PowerShell из корня проекта (Milvus запущен, `.venv` активна). Перед ними: `$env:PYTHONIOENCODING='utf-8'; [Console]::OutputEncoding=[Text.Encoding]::UTF8`.

```
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split_extra.json --mode base --allow-db-write      # опыт 1
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode base --allow-db-write            # опыт 2
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode e5 --allow-db-write              # контроль
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode rerank --allow-db-write          # опыт 3
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split_extra.json --mode rerank --reuse-collection  # опыт 4
python V2/scripts/run_pipeline.py load-clip --config V2/configs/pipeline_manual_merged_split.json --allow-db-write
python V2/scripts/run_pipeline.py all --config V2/configs/pipeline_manual_merged_split.json --mode rerank --reuse-collection --with-clip  # опыт 5
```

`--allow-db-write` останавливается, если коллекция уже есть, поэтому для повторного поиска по готовым коллекциям в ячейке ниже используется `--reuse-collection`.

In [ ]:
import subprocess, sys
CFG = {"o1": ("pipeline_manual_merged_split_extra.json", "base"), "o2": ("pipeline_manual_merged_split.json", "base"),
       "o3": ("pipeline_manual_merged_split.json", "rerank"), "o4": ("pipeline_manual_merged_split_extra.json", "rerank"),
       "o5": ("pipeline_manual_merged_split.json", "rerank")}
def run(cmd):
    print(">", " ".join(cmd)); rc = subprocess.run(cmd, cwd=str(ROOT)).returncode
    if rc: raise RuntimeError(f"код {rc}")
if RUN_LIVE:
    for e in EXP:
        cfg, mode = CFG[e]
        run([sys.executable, "V2/scripts/run_pipeline.py", "all", "--config", f"V2/configs/{cfg}", "--mode", mode, "--reuse-collection"] + (["--with-clip"] if e == "o5" else []))
elif RUN_EVAL:
    # Пересчёт метрик по сохранённой выдаче без Milvus: создаётся новая папка, старые файлы не меняются.
    for e in EXP:
        cfg, mode = CFG[e]
        run([sys.executable, "V2/scripts/run_pipeline.py", "evaluate", "--config", f"V2/configs/{cfg}", "--mode", mode,
             "--input", str((RUNS[e][0].parent / "retrieval.json").relative_to(ROOT))])
else:
    print("RUN_LIVE = False и RUN_EVAL = False — используются сохранённые metrics.json.")

## 8. Выводы (кратко; подробно — в отчёте Word, раздел 8)

1. По сравнению с первым проектом при том же зачёте по пункту и тех же моделях поиск заметно лучше; основная причина — исправленный корпус.
2. Префиксы E5 дают небольшой устойчивый выигрыш (опыт 1 → 2), заметно больше расхождения контрольного повтора.
3. Главный прирост даёт реранкер (опыт 2 → 3), 50 кандидатов лучше 20 (опыт 3 → 4).
4. CLIP в текущем виде не меняет результат (опыт 4 → 5); изображения в зачёт пока не входят.
5. Рекомендуемая конфигурация — опыт 4.
6. Генерация по выдаче опыта 4 прошла на всех 720 вопросах без сбоев (раздел 6).
7. По оценке LLM-судьи итоговый балл 93.3 % (исходные 92.4 %, части 93.9 %), все критерии на максимуме у 85.1 % ответов, критические ошибки — 19 (2.6 %), 9 из них — эталонный пункт не подан поиском (раздел 6.4).
8. На 48 вопросах без ответа в ПУЭ правильный отказ — 95.8 % по судье и 89.6 % по ручной проверке; слабое место — смежные темы (раздел 6.5). Судья — та же модель и мягче человека.

## 9. План

Ручная проверка выборки ответов основного набора и 19 критических ошибок (согласие судьи с человеком), при возможности — судья другой моделью; расширение смежных тем в наборе на отказ; оценка выдачи изображений; замер скорости; подключение к Telegram-боту; V3 — универсальный конвейер документов.